In [1]:
import random
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

seed = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

data = pd.read_csv("supplementary_data/data.csv")
feature_columns = ["O", "N", "SSA", "PV", "RMIC", "Dap", "ID/IG", "CD", "Anion"]
target = "Cs"
data["Anion"] = data["Anion"].map({"SO4": 0, "OTf": 1}).astype(int)

X = data[feature_columns]
y = data[target]
cs_bin = pd.qcut(y, q=10, labels=False, duplicates="drop")

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=cs_bin
)

train_bin = pd.qcut(y_train_full, q=10, labels=False, duplicates="drop")
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15,
    random_state=seed, stratify=train_bin
)

edge_index = torch.tensor([
    [0, 1], [1, 0], [1, 2], [1, 6],
    [2, 1], [2, 3], [3, 2], [3, 4],
    [4, 3], [4, 5], [4, 6], [5, 4],
    [6, 1], [6, 4]
], dtype=torch.long).t().contiguous()

def build_graph_from_features(features, target_scaled):
    features = torch.tensor(features, dtype=torch.float32)
    x = torch.diag(features)
    y = torch.tensor([target_scaled], dtype=torch.float32)
    return Data(x=x, edge_index=edge_index.clone(), y=y)

class CustomGraphDataset(Dataset):
    def __init__(self, X_scaled, y_scaled):
        super().__init__()
        self.graph_list = [
            build_graph_from_features(X_scaled[i], y_scaled[i])
            for i in range(len(X_scaled))
        ]

    def len(self):
        return len(self.graph_list)

    def get(self, idx):
        return self.graph_list[idx]

class GNNModel(nn.Module):
    def __init__(self, input_dim=9, hidden_dims=[63, 63, 63], output_dim=1):
        super(GNNModel, self).__init__()
        dimensions = [input_dim] + hidden_dims
        self.conv_layers = nn.ModuleList([
            GCNConv(dimensions[i], dimensions[i + 1])
            for i in range(len(hidden_dims))
        ])
        self.linear = nn.Linear(hidden_dims[-1], output_dim)

    def forward(self, data):
        x = data.x
        for conv in self.conv_layers:
            x = F.relu(conv(x, data.edge_index))
        x = global_mean_pool(x, data.batch)
        return self.linear(x).view(-1)

def create_dataset(X_values, y_values):
    return CustomGraphDataset(X_values, y_values)

def create_loader(dataset, batch_size, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        generator=generator
    )

def evaluate_scaled(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    predictions = []
    labels = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            outputs = model(batch)
            loss = criterion(outputs, batch.y)
            total_loss += loss.item() * batch.num_graphs
            predictions.append(outputs.cpu().numpy())
            labels.append(batch.y.cpu().numpy())
    total_loss /= len(loader.dataset)
    return total_loss, np.concatenate(predictions), np.concatenate(labels)

def predict_scaled(model, loader):
    model.eval()
    predictions = []
    labels = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            predictions.append(model(batch).cpu().numpy())
            labels.append(batch.y.cpu().numpy())
    return np.concatenate(predictions), np.concatenate(labels)

def calc_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return r2, rmse, mae, mape

x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_val_scaled = x_scaler.transform(X_val)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1)).ravel()

train_dataset = create_dataset(X_train_scaled, y_train_scaled)
val_dataset = create_dataset(X_val_scaled, y_val_scaled)

batch_size = 20
input_dim = 9
hidden_dims = [63, 63, 63]
output_dim = 1
learning_rate = 0.01
num_epochs = 200
patience = 30

train_loader = create_loader(train_dataset, batch_size, True, seed)
val_loader = create_loader(val_dataset, batch_size, False, seed)

set_seed(seed)
model = GNNModel(input_dim, hidden_dims, output_dim).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

best_val_loss = np.inf
best_val_r2 = -np.inf
best_epoch = 0
best_state = None
counter = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        outputs = model(batch)
        loss = criterion(outputs, batch.y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch.num_graphs
    train_loss /= len(train_loader.dataset)

    val_loss, val_preds, val_labels = evaluate_scaled(model, val_loader, criterion)
    val_r2 = r2_score(val_labels, val_preds)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_r2 = val_r2
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1}/{num_epochs}] | Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val R2: {val_r2:.4f} | "
            f"Best R2: {best_val_r2:.4f}"
        )

    if counter >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

model.load_state_dict(best_state)
print(f"Best Epoch: {best_epoch}")
print(f"Best Validation R2: {best_val_r2:.4f}")

x_scaler = StandardScaler()
X_train_full_scaled = x_scaler.fit_transform(X_train_full)
X_test_scaled = x_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_full_scaled = y_scaler.fit_transform(
    y_train_full.values.reshape(-1, 1)
).ravel()
y_test_scaled = y_scaler.transform(
    y_test.values.reshape(-1, 1)
).ravel()

train_full_dataset = create_dataset(X_train_full_scaled, y_train_full_scaled)
test_dataset = create_dataset(X_test_scaled, y_test_scaled)

train_full_loader = create_loader(train_full_dataset, batch_size, True, seed)
train_evaluation_loader = create_loader(train_full_dataset, batch_size, False, seed)
test_loader = create_loader(test_dataset, batch_size, False, seed)

set_seed(seed)
model = GNNModel(input_dim, hidden_dims, output_dim).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(best_epoch):
    model.train()
    for batch in train_full_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        outputs = model(batch)
        loss = criterion(outputs, batch.y)
        loss.backward()
        optimizer.step()

train_preds_scaled, train_labels_scaled = predict_scaled(
    model, train_evaluation_loader
)
test_preds_scaled, test_labels_scaled = predict_scaled(
    model, test_loader
)

train_preds = y_scaler.inverse_transform(
    train_preds_scaled.reshape(-1, 1)
).ravel()
train_true = y_scaler.inverse_transform(
    train_labels_scaled.reshape(-1, 1)
).ravel()
test_preds = y_scaler.inverse_transform(
    test_preds_scaled.reshape(-1, 1)
).ravel()
test_true = y_scaler.inverse_transform(
    test_labels_scaled.reshape(-1, 1)
).ravel()

train_r2, train_rmse, train_mae, train_mape = calc_metrics(train_true, train_preds)
test_r2, test_rmse, test_mae, test_mape = calc_metrics(test_true, test_preds)

print("-------- GNN --------")
print(f"Train set R2   : {train_r2:.4f}")
print(f"Test set R2    : {test_r2:.4f}")
print(f"Train set RMSE : {train_rmse:.4f}")
print(f"Test set RMSE  : {test_rmse:.4f}")
print(f"Train set MAE  : {train_mae:.4f}")
print(f"Test set MAE   : {test_mae:.4f}")
print(f"Train set MAPE : {train_mape:.2f}%")
print(f"Test set MAPE  : {test_mape:.2f}%")

Using device: cpu
Epoch [10/200] | Train Loss: 0.3398 | Val Loss: 0.3726 | Val R2: 0.6549 | Best R2: 0.6684
Epoch [20/200] | Train Loss: 0.2532 | Val Loss: 0.2246 | Val R2: 0.7919 | Best R2: 0.7919
Epoch [30/200] | Train Loss: 0.1871 | Val Loss: 0.2033 | Val R2: 0.8117 | Best R2: 0.8117
Epoch [40/200] | Train Loss: 0.1591 | Val Loss: 0.2309 | Val R2: 0.7861 | Best R2: 0.8471
Epoch [50/200] | Train Loss: 0.1577 | Val Loss: 0.1511 | Val R2: 0.8600 | Best R2: 0.8688
Epoch [60/200] | Train Loss: 0.1266 | Val Loss: 0.1352 | Val R2: 0.8748 | Best R2: 0.8841
Epoch [70/200] | Train Loss: 0.1291 | Val Loss: 0.1177 | Val R2: 0.8910 | Best R2: 0.8910
Epoch [80/200] | Train Loss: 0.0963 | Val Loss: 0.1472 | Val R2: 0.8637 | Best R2: 0.8933
Epoch [90/200] | Train Loss: 0.0984 | Val Loss: 0.1042 | Val R2: 0.9035 | Best R2: 0.9035
Epoch [100/200] | Train Loss: 0.1064 | Val Loss: 0.1154 | Val R2: 0.8931 | Best R2: 0.9143
Epoch [110/200] | Train Loss: 0.0903 | Val Loss: 0.1193 | Val R2: 0.8895 | Best R